# Laboratory 4: Hybrid Quantum-Classical Algorithms
## The NISQ Era: VQE and QAOA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Quantum-Plus-Labs/quantum-labs-2026/blob/main/lab4/lab4_hybrid.ipynb)

*Noisy Intermediate-Scale Quantum (NISQ): today's processors, with tens to hundreds of qubits and noise that limits circuit depth.*

**Objective:** Explore the algorithms currently used on real, noisy quantum computers to solve optimization and physics problems.

**Learning Objectives:**
1. Master the **Parameter-Shift Rule** for quantum gradients.
2. Implement the **Variational Quantum Eigensolver (VQE)** for a two-qubit Ising model (a simple energy function built from $Z$'s, defined in Exercise 2).
3. Use the **Quantum Approximate Optimization Algorithm (QAOA)** for MaxCut.
4. Understand the role of the classical optimizer in the hybrid loop.

---

### Before Exercise 1: the variational loop

Every algorithm in this lab runs the same loop. A short quantum circuit with adjustable rotation angles $\theta$ prepares a state; a classical program reads the measurement results and adjusts the angles:

1. choose angles $\theta$;
2. run the circuit and measure it many times (each run is one *shot*);
3. average the results into one number, the **cost** $C(\theta)$;
4. a classical **optimizer** proposes new angles that should lower $C$. Repeat until $C$ stops decreasing.

- **$Z$ as a score.** From Lab 1, $Z=\mathrm{diag}(1,-1)$: it gives the value $+1$ to $|0\rangle$ and $-1$ to $|1\rangle$. $Z_i$ means $Z$ acting on qubit $i$, and $Z_0Z_1$ multiplies the two values: $+1$ if the two bits are equal, $-1$ if they differ.
- **Hamiltonian $H$ (the energy function).** A sum of such terms that gives every bit string a number, its *energy*. Exercise 2 uses $H=Z_0Z_1+0.5\,Z_0$: $|00\rangle\to 1.5$, $|01\rangle\to -0.5$, $|10\rangle\to -1.5$, $|11\rangle\to 0.5$ (first symbol = qubit 0). A problem is encoded so that its best answer is the bit string of lowest energy, the **ground state**, with energy $E_0$ (here $-1.5$, at $|10\rangle$). Because these Hamiltonians contain only $Z$'s, each bit string has one definite energy (in matrix language, it is an eigenvector of $H$ and the energy is its eigenvalue).
- **Expectation value $\langle H\rangle$ (the cost).** A superposition has no single energy: each shot returns one bit string $x$, with probability $P(x)$, and its energy $E(x)$. The average over many shots is $\langle H\rangle=\sum_x P(x)\,E(x)$. This is the cost $C(\theta)$. For one qubit, $\langle Z\rangle = P(0)-P(1)$.
- **Why minimising works.** An average can never be lower than the smallest value it averages, so $C(\theta)\ge E_0$ for every $\theta$, with equality only when all the probability sits on the ground state (the *variational principle*). Pushing $C$ down pushes the state toward the answer.
- **Ansatz $U(\theta)$.** German for "educated guess": a fixed circuit layout whose rotation angles are the adjustable parameters. The rotation used here is $R_y(\theta)$, which turns $|0\rangle$ into $\cos\tfrac{\theta}{2}|0\rangle+\sin\tfrac{\theta}{2}|1\rangle$; so $R_y(0)$ does nothing and $R_y(\pi)|0\rangle=|1\rangle$.
- **Classical optimizer.** Ordinary numerical software, nothing quantum. *Gradient-based* optimizers follow the slope downhill and need the gradient, which Exercise 1 shows how to obtain from the quantum computer. *Gradient-free* optimizers such as COBYLA (Constrained Optimization BY Linear Approximations, used in Exercises 2 and 3) only compare cost values.

*On real hardware each $C(\theta)$ is estimated from a finite number of shots and carries noise; the simulator in this notebook returns exact values.*

### Environment Setup

Installs Qiskit and its simulator (about a minute on Colab). The `Requirement already satisfied` lines are normal.

In [ ]:
!pip install -q qiskit[visualization] qiskit-aer qiskit-ibm-runtime matplotlib scipy pylatexenc

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit.visualization import plot_histogram
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE!")

### Exercise 1: The Parameter-Shift Rule

**Why this step:** a variational algorithm improves its angles by following the slope (gradient) of the cost. A quantum computer returns only *values* of the cost, not its formula, so we need a way to get the slope from values alone.

The circuit below applies $R_y(\theta)$ to $|0\rangle$, giving $\cos\tfrac{\theta}{2}|0\rangle+\sin\tfrac{\theta}{2}|1\rangle$, and measures the average of $Z$: $f(\theta)=\langle Z\rangle=P(0)-P(1)=\cos\theta$. Its exact derivative is therefore $-\sin\theta$ (the "analytic gradient" in the code). The **parameter-shift rule** obtains the same number from two circuit evaluations at shifted angles:
$$\frac{\partial f}{\partial\theta} = \frac{f(\theta + \pi/2) - f(\theta - \pi/2)}{2}$$

**Why $\pi/2$?** Any shift $s$ works: $f'(\theta) = \dfrac{f(\theta+s) - f(\theta-s)}{2\sin s}$. $\pi/2$ is the choice that makes the factor $\tfrac12$ and keeps the two evaluations furthest apart.

Each evaluation is an average: on hardware it needs many shots; the simulator used here returns the exact average.

**Your Task:**
Run the cell: at $\theta=\pi/4$ both numbers are $-0.707107$, difference $0$.

In [ ]:
def evaluate_circuit(theta, estimator):
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    obs = SparsePauliOp('Z')
    return estimator.run([(qc, obs)]).result()[0].data.evs

estimator = EstimatorV2()
theta_val = np.pi/4

f_plus = evaluate_circuit(theta_val + np.pi/2, estimator)
f_minus = evaluate_circuit(theta_val - np.pi/2, estimator)
psr_grad = (f_plus - f_minus) / 2
analytic_grad = -np.sin(theta_val)

print(f"PSR Gradient: {psr_grad:.6f}")
print(f"Analytic Gradient: {analytic_grad:.6f}")
print(f"Difference: {abs(psr_grad - analytic_grad):.2e}")

### Exercise 2: VQE on the Ising Model

**Why this step:** VQE = Variational Quantum Eigensolver. We look for the lowest energy of $H = Z_0Z_1 + 0.5\,Z_0$, the same example as Part B of the guide. Its four bit strings have energies $|00\rangle{:}\,1.5$, $|01\rangle{:}\,{-0.5}$, $|10\rangle{:}\,{-1.5}$, $|11\rangle{:}\,0.5$ (first symbol = qubit 0), so the target is $E_0=-1.5$ at $|10\rangle$. The circuit (the *ansatz*) is $R_y(\theta_0)$ on qubit 0, $R_y(\theta_1)$ on qubit 1, then a CNOT. The estimator returns the average energy for given angles, and COBYLA, a classical optimizer that does not need gradients, adjusts the two angles to lower it.

*Code note:* Qiskit writes Pauli labels with qubit 0 on the **right** (little-endian), so $0.5\,Z_0$ is `'IZ'`; `'ZZ'` is the same either way.

**Expected output:** VQE energy −1.5000, final angles ≈ (3.142, 3.142), the crosshair of the guide's heatmap.

In [ ]:
from qiskit.circuit import ParameterVector

def build_ansatz():
    theta = ParameterVector('θ', 2)
    qc = QuantumCircuit(2)
    qc.ry(theta[0], 0)
    qc.ry(theta[1], 1)
    qc.cx(0, 1)
    return qc, theta

hamiltonian = SparsePauliOp.from_list([('ZZ', 1.0), ('IZ', 0.5)])
ansatz, params = build_ansatz()
history = []

def cost_func(x):
    res = estimator.run([(ansatz, hamiltonian, x)]).result()[0].data.evs
    history.append(res)
    return res

x0 = np.random.rand(2)
res = minimize(cost_func, x0, method='COBYLA')

print(f"Target Energy: -1.5")
print(f"VQE Energy: {res.fun:.4f}")
print(f"Final angles: θ0 = {res.x[0]:.3f}, θ1 = {res.x[1]:.3f}  (guide: π, π)")

plt.plot(history)
plt.axhline(-1.5, color='r', linestyle='--')
plt.xlabel('Cost evaluation')
plt.ylabel('Energy')
plt.title('VQE Convergence')
plt.show()

### Exercise 3: QAOA for MaxCut on the 4-cycle

**Why this step:** VQE searched for the lowest energy of a physics model. QAOA uses the same loop to solve a combinatorial problem: split the 4 nodes of a ring into two groups so that as many edges as possible join the two groups ("cut" edges). The cost is the average number of cut edges, $\langle H_C\rangle$ with $H_C=\sum_{\text{edges}}(I-Z_iZ_j)/2$, since $(1-Z_iZ_j)/2$ is 1 for a cut edge and 0 otherwise. Each layer adds two angles $(\gamma,\beta)$; COBYLA tunes them.

**Expected output:** ratio 0.750 at $p=1$, 1.000 at $p=2$ and $p=3$. On this tiny graph two layers already put every shot on an optimal cut; this is not typical of large graphs.

In [ ]:
from qiskit.circuit import ParameterVector

def build_qaoa(n, edges, p):
    """QAOA circuit: H on all qubits, then p layers of (cost, mixer)."""
    g = ParameterVector('γ', p)
    b = ParameterVector('β', p)
    qc = QuantumCircuit(n)
    qc.h(range(n))                      # start: all 16 splits equally likely
    for k in range(p):
        for (u, v) in edges:            # cost layer: phase e^{-iγ Z_u Z_v} per edge
            qc.cx(u, v)
            qc.rz(2 * g[k], v)
            qc.cx(u, v)
        qc.rx(2 * b[k], range(n))       # mixer: R_x(2β) on every qubit
    return qc

def maxcut_hamiltonian(n, edges):
    """H_C = sum over edges of (I - Z_u Z_v)/2 = number of cut edges."""
    terms = []
    for (u, v) in edges:
        zz = ['I'] * n
        zz[u] = 'Z'; zz[v] = 'Z'
        terms += [(''.join(reversed(zz)), -0.5), ('I' * n, 0.5)]
    return SparsePauliOp.from_list(terms).simplify()

edges = [(0, 1), (1, 2), (2, 3), (3, 0)]    # the 4-cycle C4; best cut = 4
H_C = maxcut_hamiltonian(4, edges)
for p in [1, 2, 3]:
    qc = build_qaoa(4, edges, p)
    def neg_cut(x):                          # COBYLA minimises, so negate
        return -estimator.run([(qc, H_C, x)]).result()[0].data.evs
    res = minimize(neg_cut, [0.5] * (2 * p), method='COBYLA')
    print(f"p={p}: average cut = {-res.fun:.3f} of 4   ratio = {-res.fun/4:.3f}")

### Optional: from MaxCut to the portfolio problem (Lab 4 Finance)

*Optional preview. Lab 4 Finance does not require this notebook.*

The portfolio problem of Lab 4 Finance is a QUBO (Quadratic Unconstrained Binary Optimization): $\min_{x \in \{0,1\}^n} x^T Q x$. Substituting $x_i = (1 - Z_i)/2$ turns it into a Hamiltonian built from $Z$'s, exactly as MaxCut became $H_C$ in Exercise 3.

| | MaxCut (Exercise 3) | Portfolio (Lab 4 Finance) |
|---|---|---|
| Binary variables | node $\to$ group $\{0,1\}$ | asset selected or not $\{0,1\}$ |
| Objective | maximise $\sum_{(i,j) \in E} (x_i - x_j)^2$, the number of cut edges | minimise $\lambda\, x^T\Sigma x - \mu^T x$ (risk minus return) plus a budget penalty |
| $Z_iZ_j$ coefficients | $-1/2$ on every edge | $Q_{ij}/2$ on every asset pair (from the covariance $\Sigma$, plus the penalty) |
| $Z_i$ coefficients | none | $h_i$ (from $\mu$, $\Sigma$ and the penalty) |
| Cost layer | CNOT–$R_z$–CNOT per edge | CNOT–$R_z$–CNOT per asset pair, plus an $R_z$ per asset |
| Mixer | $R_x(2\beta)$ | $R_x(2\beta)$, identical |

Only the Hamiltonian $H_C$ changes: the QAOA circuit structure and the optimization loop stay the same. COBYLA, used here, is gradient-free: it only compares cost values. A gradient-based optimizer would take its slopes from the parameter-shift rule of Exercise 1.